# Job Match — Upload Resume + Job Description → Skill Gap

A single self-contained notebook: upload a resume, paste or upload a job description, click **Analyze**, and see the skill gap right below — no separate web app, no server port, no proxy setup. Everything runs inside the notebook's own kernel using `ipywidgets`, the same widget type you've already used for resume uploads.

Run every cell top to bottom once, then use the widgets in the last cell as many times as you like — no need to re-run earlier cells between analyses.


## Step 1 — Setup


In [1]:
!pip install pdfplumber python-docx spacy ipywidgets --quiet
!python -m spacy download en_core_web_sm --quiet


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import io
import re
import json
import sqlite3
import pdfplumber
import spacy
import ipywidgets as widgets
from docx import Document
from spacy.matcher import PhraseMatcher
from IPython.display import display, HTML, clear_output

DB_PATH = "careerpilot.db"

nlp = spacy.load("en_core_web_sm")

# Expanded skill vocabulary — covers languages, frameworks, cloud/devops,
# data/ML, databases, and common tools/methodologies. Add to this list
# freely; it's the single source of truth for what "counts" as a skill.
SKILLS = [
    # Languages
    "Python", "Java", "JavaScript", "TypeScript", "C++", "C#", "Go", "Rust",
    "Scala", "R", "Kotlin", "Swift", "PHP", "Ruby", "SQL", "Bash",
    # Web / frameworks
    "React", "Angular", "Vue.js", "Node.js", "Express.js", "Django",
    "Flask", "FastAPI", "Spring Boot", "Next.js", "HTML", "CSS",
    "Tailwind CSS", "Redux", "GraphQL", "REST API",
    # Data / ML / AI
    "Machine Learning", "Deep Learning", "NLP", "Computer Vision",
    "Data Analysis", "Data Engineering", "TensorFlow", "PyTorch",
    "Keras", "Scikit-learn", "Pandas", "NumPy", "OpenCV", "spaCy",
    "Hadoop", "Spark", "Tableau", "Power BI",
    # Databases
    "MongoDB", "PostgreSQL", "MySQL", "Redis", "Elasticsearch",
    "SQLite", "DynamoDB", "Cassandra",
    # Cloud / DevOps
    "AWS", "Azure", "GCP", "Docker", "Kubernetes", "Terraform",
    "Ansible", "Jenkins", "CI/CD", "Linux", "Nginx", "Git",
    # Practices / testing / tools
    "Agile", "Scrum", "Microservices", "JUnit", "Pytest", "Selenium",
    "Jira", "Confluence",
]

matcher = PhraseMatcher(nlp.vocab, attr="LOWER")
matcher.add("SKILLS", [nlp.make_doc(s) for s in SKILLS])

print(f"Setup complete. Tracking {len(SKILLS)} skills.")

Setup complete. Tracking 78 skills.


## Step 2 — Parsing and skill-matching functions (reused from the main pipeline)


In [3]:
def extract_text_from_bytes(filename: str, data: bytes) -> str:
    name = filename.lower()
    if name.endswith(".pdf"):
        text = ""
        with pdfplumber.open(io.BytesIO(data)) as pdf:
            for page in pdf.pages:
                text += (page.extract_text() or "") + "\n"
        return text
    if name.endswith(".docx"):
        doc = Document(io.BytesIO(data))
        parts = [p.text for p in doc.paragraphs]
        # Paragraphs alone miss anything laid out in a table (skills lists,
        # experience blocks, etc. are often tables in resume templates).
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    if cell.text.strip():
                        parts.append(cell.text)
        return "\n".join(parts)
    return data.decode("utf-8", errors="ignore")   # .txt fallback

def get_skills(text: str) -> list:
    doc = nlp(text)
    matches = matcher(doc)
    found = {doc[start:end].text for _, start, end in matches}
    canonical = {s.lower(): s for s in SKILLS}
    return sorted({canonical.get(f.lower(), f) for f in found})

def get_name(text: str) -> str:
    doc = nlp(text[:1000])
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            return ent.text
    # Fallback: resume headers often aren't recognized as PERSON entities
    # by spaCy's small model (odd fonts, all-caps, no surrounding context).
    # Try the first non-empty line if it plausibly looks like a name.
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        words = line.split()
        if (1 < len(words) <= 4
                and all(w.replace("-", "").replace(".", "").isalpha() for w in words)
                and "@" not in line):
            return line
        break
    return "Candidate"

def get_email(text: str) -> str:
    match = re.search(r"[\w.+-]+@[\w-]+\.[\w.-]+", text)
    return match.group(0) if match else ""

print("Functions ready.")

Functions ready.


## Step 3 — (Optional) Save each analyzed resume to `careerpilot.db`

Keeps this notebook connected to the same database the rest of your pipeline (matching, Ollama explanations) already uses.


In [4]:
def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS career_profiles (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            filename TEXT UNIQUE, name TEXT, email TEXT, phone TEXT, location TEXT,
            education TEXT, skills TEXT, experience TEXT, organizations TEXT,
            certifications TEXT, career_goal TEXT, profile_completion INTEGER,
            raw_text TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    conn.close()

def save_profile(profile: dict):
    conn = sqlite3.connect(DB_PATH)
    # Upsert on filename: re-analyzing the same resume updates its row
    # instead of piling up duplicate entries every time Analyze is clicked.
    conn.execute("""
        INSERT INTO career_profiles
        (filename, name, email, phone, location, education, skills, experience,
         organizations, certifications, career_goal, profile_completion, raw_text)
        VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)
        ON CONFLICT(filename) DO UPDATE SET
            name=excluded.name, email=excluded.email, phone=excluded.phone,
            location=excluded.location, education=excluded.education,
            skills=excluded.skills, experience=excluded.experience,
            organizations=excluded.organizations, certifications=excluded.certifications,
            career_goal=excluded.career_goal, profile_completion=excluded.profile_completion,
            raw_text=excluded.raw_text
    """, (
        profile.get("filename", ""), profile.get("name", ""), profile.get("email", ""),
        profile.get("phone", ""), profile.get("location", ""),
        json.dumps(profile.get("education", [])), json.dumps(profile.get("skills", [])),
        json.dumps(profile.get("experience", [])), json.dumps(profile.get("organizations", [])),
        json.dumps(profile.get("certifications", [])), json.dumps(profile.get("career_goal", {})),
        profile.get("profile_completion", 0), profile.get("raw_text", ""),
    ))
    conn.commit()
    conn.close()

init_db()
print("Database ready.")

Database ready.


## Step 4 — Ollama setup (for "Why this match?" and "Get Roadmap")

These two features are LLM calls, so they're gated behind explicit buttons — the skill gap itself never waits on this. If a model is slow or times out, the core result above still works.


In [5]:
import requests

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen3.5:4b"   # swap to qwen3.5:0.8b or qwen3.5:2b if this is too slow
OLLAMA_TIMEOUT = 180

def call_ollama(prompt: str) -> str:
    try:
        response = requests.post(OLLAMA_URL, json={
            "model": OLLAMA_MODEL, "prompt": prompt, "stream": False,
        }, timeout=OLLAMA_TIMEOUT)
    except requests.exceptions.ConnectionError:
        return ("[Could not reach Ollama at " + OLLAMA_URL + ". "
                "Is `ollama serve` running, and is the model pulled? "
                "Try: ollama pull " + OLLAMA_MODEL + "]")
    except requests.exceptions.Timeout:
        return f"[Ollama timed out after {OLLAMA_TIMEOUT}s. Try a smaller model.]"
    except requests.exceptions.RequestException as e:
        return f"[Ollama request failed: {e}]"

    if response.status_code != 200:
        return f"[Ollama error: {response.text}]"
    return response.json().get("response", "").strip()

print("Ollama helper ready.")

Ollama helper ready.


## Step 5 — Select resume + analyze

**Resume selection uses a dropdown instead of a live upload button** — `ipywidgets.FileUpload` has been unreliable in this environment (same issue we hit earlier in this project), so this avoids it entirely:

1. Drag your resume file into the `uploads/` folder using Jupyter's own file browser on the left (the same way you've uploaded resumes before)
2. Click **↻ Refresh file list** below so it shows up in the dropdown
3. Select it, paste/upload a job description, click **Analyze**

Run this cell once — it stays live and reusable; click Analyze as many times as you want without re-running anything above.


In [6]:
import os

UPLOAD_DIR = "uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# --- Resume: pick from a dropdown instead of a live browser upload ---
# (ipywidgets.FileUpload has been unreliable in this environment — see earlier debugging.
#  This avoids it entirely: drag your resume into the uploads/ folder using Jupyter's own
#  file browser on the left, then refresh and pick it below.)
def list_resume_files():
    files = [f for f in os.listdir(UPLOAD_DIR) if f.lower().endswith((".pdf", ".docx", ".txt"))]
    return files if files else ["(no files found — upload one to uploads/ first)"]

resume_dropdown = widgets.Dropdown(options=list_resume_files(), description="Resume:")
refresh_button = widgets.Button(description="↻ Refresh file list")

def on_refresh_click(b):
    resume_dropdown.options = list_resume_files()

refresh_button.on_click(on_refresh_click)

job_uploader = widgets.FileUpload(accept=".pdf,.docx,.txt", multiple=False, description="JD file")
job_text_area = widgets.Textarea(placeholder="...or paste the job description here instead", layout=widgets.Layout(width="600px", height="150px"))
job_title_box = widgets.Text(placeholder="Job title (optional, for display)", layout=widgets.Layout(width="400px"))
analyze_button = widgets.Button(description="Analyze", button_style="success")
save_checkbox = widgets.Checkbox(value=True, description="Save resume to careerpilot.db")
output = widgets.Output()

# Buttons that only make sense after Analyze has run at least once
why_button = widgets.Button(description="🤔 Why this match?", button_style="info")
roadmap_button = widgets.Button(description="🗺️ Get Roadmap", button_style="warning")
why_output = widgets.Output()
roadmap_output = widgets.Output()

# Holds the results of the last Analyze click, so the Why/Roadmap buttons have something
# to work with without recomputing anything.
session = {}


def get_job_description_text() -> str:
    if len(job_uploader.value) > 0:
        item = job_uploader.value[0] if isinstance(job_uploader.value, tuple) else list(job_uploader.value.values())[0]
        name = item["name"] if isinstance(job_uploader.value, tuple) else list(job_uploader.value.keys())[0]
        content = item["content"]
        return extract_text_from_bytes(name, bytes(content))
    return job_text_area.value


def on_analyze_click(b):
    with output:
        clear_output()

        selected_file = resume_dropdown.value
        if not selected_file or selected_file.startswith("(no files"):
            print("No resume selected. Drag a file into the uploads/ folder, then click Refresh.")
            return

        job_description_text = get_job_description_text()
        if not job_description_text.strip():
            print("Provide a job description — paste text or upload a file.")
            return

        resume_path = os.path.join(UPLOAD_DIR, selected_file)
        with open(resume_path, "rb") as f:
            resume_bytes = f.read()
        resume_text = extract_text_from_bytes(selected_file, resume_bytes)

        name = get_name(resume_text)
        email = get_email(resume_text)
        resume_skills = set(get_skills(resume_text))
        job_skills = set(get_skills(job_description_text))

        matched = sorted(resume_skills & job_skills)
        missing = sorted(job_skills - resume_skills)
        extra = sorted(resume_skills - job_skills)
        match_pct = round(len(matched) / len(job_skills) * 100) if job_skills else 0

        if save_checkbox.value:
            save_profile({
                "filename": selected_file, "name": name, "email": email,
                "skills": sorted(resume_skills), "raw_text": resume_text,
                "profile_completion": 60,
            })

        title = job_title_box.value or "this role"

        # save for the Why/Roadmap buttons
        session.clear()
        session.update(dict(
            name=name, title=title, job_description_text=job_description_text,
            matched=matched, missing=missing, extra=extra, match_pct=match_pct,
        ))

        display(HTML(f"""
            <h3>Results for {name} — {title}</h3>
            <p><b>Skill match: {match_pct}%</b></p>
            <p>✅ <b>Matched:</b> {", ".join(matched) if matched else "none"}</p>
            <p>❌ <b>Missing:</b> {", ".join(missing) if missing else "none — full coverage"}</p>
            <p>➕ <b>Extra (not required):</b> {", ".join(extra) if extra else "none"}</p>
        """))


def on_why_click(b):
    with why_output:
        clear_output()
        if not session:
            print("Click Analyze first.")
            return
        print("Generating explanation... (can take up to a minute or two)")
        prompt = (
            f"A candidate applied for '{session['title']}'. \n"
            f"Matched required skills: {', '.join(session['matched']) or 'none'}.\n"
            f"Missing required skills: {', '.join(session['missing']) or 'none'}.\n"
            f"Overall skill match: {session['match_pct']}%.\n\n"
            "In 3-4 sentences, explain clearly why this match score makes sense: what "
            "specifically is working in the candidate's favor, and what specifically is "
            "holding the score back. Be concrete about which skills matter most and why, "
            "not just a restatement of the lists above."
        )
        explanation = call_ollama(prompt)
        clear_output()
        display(HTML(f"<p><b>Why this match:</b> {explanation}</p>"))


def on_roadmap_click(b):
    with roadmap_output:
        clear_output()
        if not session:
            print("Click Analyze first.")
            return
        if not session["missing"]:
            print("No skill gaps — nothing to build a roadmap for. You're already covered.")
            return
        print("Generating roadmap... (can take up to a minute or two)")
        prompt = (
            f"A candidate wants '{session['title']}' and is missing these skills, in no "
            f"particular order: {', '.join(session['missing'])}.\n"
            f"They already know: {', '.join(session['matched']) or 'nothing relevant yet'}.\n\n"
            "Write a short learning roadmap:\n"
            "1. Put the missing skills in the best learning order, and say briefly why that "
            "order makes sense (e.g. one skill is a prerequisite for another).\n"
            "2. For each skill, suggest ONE realistic way to learn it — name a type of "
            "resource (official documentation, a well-known free course platform, a "
            "hands-on project idea) rather than inventing a specific URL you can't verify.\n"
            "3. Give a rough total time estimate to close the gap for someone learning "
            "part-time.\n"
            "Keep it concise and practical, plain text, no markdown headers."
        )
        roadmap = call_ollama(prompt)
        clear_output()
        display(HTML(f"<p><b>Roadmap:</b></p><pre style='white-space: pre-wrap'>{roadmap}</pre>"))


analyze_button.on_click(on_analyze_click)
why_button.on_click(on_why_click)
roadmap_button.on_click(on_roadmap_click)

display(widgets.VBox([
    widgets.HTML("<b>1. Select your resume</b> (drag it into the uploads/ folder using the file browser on the left first)"),
    widgets.HBox([resume_dropdown, refresh_button]),
    widgets.HTML("<b>2. Job description</b>"), job_title_box, job_text_area,
    widgets.HTML("<i>— or —</i>"), job_uploader,
    save_checkbox,
    analyze_button,
    output,
    widgets.HTML("<hr><b>3. Dig deeper</b> (run Analyze first)"),
    widgets.HBox([why_button, roadmap_button]),
    why_output,
    roadmap_output,
]))
